> **Renumbered 2026-10-08.** This notebook is **HSMM 06** (formerly notebook/Module 12). Text below and in the discussions may still use the old global numbers; see the mapping in `notebooks/README.md`.


# 12 — Supervised volatility forecasts: HAR null, implied variance and HAR-X

**Design v0.2.** This revision fixes the two comparison flaws identified after the first real-data run (discussions/006–007):

1. Ridge penalties are selected with a **paired per-session one-standard-error rule** using the loss difference versus the best penalty, not the cross-session variability of the loss level.
2. Every nested comparison is fitted on the **larger model's complete rows**. Previously selected block penalties are frozen and only the newly added block receives a new penalty. Therefore each hierarchy step is a pure incremental-information test.

Backward HAR windows use a pre-registered **90% minimum return coverage** rule with variance sums rescaled by k / n_present. Future 30m/60m targets remain strict full-count targets.

The old 06_frozen_spec.json belongs to design v0.1 and is invalid for confirmation. This notebook writes and reads 06_frozen_spec_v02.json. The binding test remains the first 30 completed sessions after 2026-10-07.

In [1]:
from pathlib import Path
import os, sys, time, json, subprocess, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 1000)

load_dotenv()
DATA_ROOT = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve()
DB_PATH = DATA_ROOT / "database" / "research.duckdb"
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "data_engineering").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "data_engineering").exists():
    raise RuntimeError("Repository root not found from current working directory.")
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

for _m in [m for m in sys.modules if m.split(".")[0] == "data_engineering"]:
    del sys.modules[_m]

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling import full_information as fi
from data_engineering.modeling import vol_targets as vt
from data_engineering.modeling import vol_har as vh
from data_engineering.modeling.clock_baseline import CLOCK_BASELINE_VERSION
from data_engineering.modeling.hsmm.design import build_design
from data_engineering.modeling.hsmm.selection import blocked_folds

if vt.VOL_TARGETS_VERSION < "0.1" or vh.VOL_HAR_VERSION < "0.1":
    raise RuntimeError("Old Module 12 code: git pull and restart the kernel.")

DESIGN_VERSION = "0.2"
MODE = os.getenv("NB12_MODE", "confirmation")
ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
DEV_START, SCREEN_TRAIN_END, DEV_END = "2026-09-22", "2026-09-28", "2026-10-07"
MIN_FEATURE_COVERAGE = float(os.getenv("NB12_MIN_FEATURE_COVERAGE", "0.45"))
BACK_MIN_COVERAGE = float(os.getenv("NB12_BACK_MIN_COVERAGE", "0.90"))
N_BOOT = int(os.getenv("NB12_BOOT", "2000"))
FREEZE = os.getenv("NB12_FREEZE", "1") == "1"
REFREEZE = os.getenv("NB12_REFREEZE", "0") == "1"
OUT = PROJECT_ROOT / "notebooks" / "HSMM" / "06"   # outputs live in a numbered subfolder
OUT.mkdir(parents=True, exist_ok=True)
SPEC_PATH = OUT / "06_frozen_spec_v02.json"
CONFIRM_MIN_SESSIONS = 30
HORIZONS = list(vt.HORIZONS)

print(
    f"mode {MODE} | design {DESIGN_VERSION} | vol_targets {vt.VOL_TARGETS_VERSION} | "
    f"vol_har {vh.VOL_HAR_VERSION} | clock {CLOCK_BASELINE_VERSION} | "
    f"development {DEV_START} -> {DEV_END} (screening fit through {SCREEN_TRAIN_END}) | "
    f"backward min coverage {BACK_MIN_COVERAGE:.0%} | lambda grid {vh.LAMBDA_GRID}"
)

mode confirmation | design 0.2 | vol_targets 0.1 | vol_har 0.1 | clock 0.2 | development 2026-09-22 -> 2026-10-07 (screening fit through 2026-09-28) | backward min coverage 90% | lambda grid (0.0, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0)


## 1. Load the state table

In development mode only sessions up to 2026-10-07 are read: confirmation data is never touched before the specification is frozen.

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])

required = {
    "state_id", "state_time", "underlying", "return_5m", "futures_index_price", "is_weekend",
    "in_settlement_window", "vwap", "volume_usd", "futures_tradable_mid_price", "trade_count",
    "perp_integrity_ok", "known_exchange_event", "integrity_flag_trades_outside_bbo",
    "microstructure_features_ready", "instability_features_ready", "option_snapshot_ready",
    "option_atm_quote_ok", "option_shape_quote_ok", "fit_ok",
    "option_iv_atm_forward",
}
optional = {"open", "high", "low", "close"}            # perp trade OHLC: range regressors only
missing_required = sorted(required - cols)
if missing_required:
    raise RuntimeError(f"time_series_feature_state lacks {missing_required}")
want = sorted((required | optional | {s.column for s in fi.FEATURE_SPECS}) & cols)
if optional - cols:
    print(f"Perp OHLC columns missing ({sorted(optional - cols)}): Parkinson / Garman-Klass regressors will be empty.")

DEV_LAST_BAR = pd.Timestamp(f"{DEV_END} 08:00", tz="UTC")     # closes the last development expiry
dev_filter = "" if MODE == "confirmation" else (
    f"WHERE state_time <= TIMESTAMPTZ '{DEV_LAST_BAR.isoformat()}'"
)
df = con.execute(
    f"SELECT {', '.join(want)} FROM time_series_feature_state {dev_filter} ORDER BY underlying, state_time"
).df()
con.close()

df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
df = df[df["underlying"].isin(ASSETS)].reset_index(drop=True)
df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["session_date"] = ri.session_key(df["state_time"])
if not (df["session_date"].to_numpy() == vt.expiry_time(df["state_time"]).sub(pd.Timedelta(hours=8)).dt.strftime("%Y-%m-%d").to_numpy()).all():
    raise RuntimeError("Session keys of regime_inputs and vol_targets disagree.")

if MODE != "confirmation":
    # The bar closing at DEV_END 08:00 carries the last return of the final development
    # expiry; it is kept only as a target increment (its session is never an origin).
    keep = ((df["session_date"] >= DEV_START) & (df["session_date"] <= DEV_END)) | (df["state_time"] == DEV_LAST_BAR)
    df = df[keep].reset_index(drop=True)
    assert df["state_time"].max() <= DEV_LAST_BAR, "confirmation data leaked into development"

DEV_SESSIONS = sorted(s for s in df["session_date"].unique() if DEV_START <= s <= DEV_END)
SCREEN_TRAIN = {s for s in DEV_SESSIONS if s <= SCREEN_TRAIN_END}
SCREEN_EVAL = {s for s in DEV_SESSIONS if s > SCREEN_TRAIN_END}
print(
    f"Rows {len(df):,} | development sessions {len(DEV_SESSIONS)} "
    f"(screening fit {len(SCREEN_TRAIN)}, screening evaluation {len(SCREEN_EVAL)})"
)
if len(SCREEN_TRAIN) < 5 or len(SCREEN_EVAL) < 3:
    raise RuntimeError("Module 12 needs the Sep 22-28 sessions and the later development sessions.")

Rows 84,672 | development sessions 16 (screening fit 7, screening evaluation 9)


## 2. Inputs per asset

* **Future targets stay strict.** Every forward return in the 30m/60m target must be present and contiguous.
* **Backward HAR windows use at least 90% observed returns**, rescaled by k / n_present. With the current windows this still requires 6/6 returns at 30m, at least 11/12 at 1h, and at least 44/48 at 4h.
* D features remain origin-time causal.
* Feature eligibility is decided on training rows only.
* Every nested comparison is fitted on identical rows within the pair.

In [3]:
def asset_frame(asset, frame=None):
    frame = df if frame is None else frame
    return frame[frame["underlying"] == asset].sort_values("state_time").reset_index(drop=True)


def backward_sum_coverage(x, times, k, min_fraction=BACK_MIN_COVERAGE):
    x = np.asarray(x, float)
    n = len(x)
    out = np.full(n, np.nan)
    if n < k:
        return out

    j = np.arange(k - 1, n)
    i = j - (k - 1)
    t = pd.to_datetime(times, utc=True).to_numpy()
    contiguous = (t[j] - t[i]) == (j - i) * vt.BAR.to_timedelta64()

    finite = np.isfinite(x)
    csum = np.r_[0.0, np.cumsum(np.where(finite, x, 0.0))]
    ccnt = np.r_[0, np.cumsum(finite.astype(int))]
    s = csum[j + 1] - csum[i]
    cnt = ccnt[j + 1] - ccnt[i]

    need = int(np.ceil(min_fraction * k))
    valid = contiguous & (cnt >= need) & (cnt > 0)
    scaled = np.where(cnt > 0, s * (k / np.maximum(cnt, 1)), np.nan)
    out[j] = np.where(valid, scaled, np.nan)
    return out


def build_panel_v02(g):
    panel = vt.build_panel(g)
    r2 = panel.r ** 2
    panel.back_rv = {
        w: backward_sum_coverage(r2, panel.times, k, BACK_MIN_COVERAGE)
        for w, k in vt.BACK_WINDOWS.items()
    }
    return panel


def d_features(g, nuis):
    cbs = nuis.clock_set()
    clock_frame = g[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]]
    f = cbs.transform(clock_frame)["return_5m__clock_vol_factor"].fillna(1.0).to_numpy()
    d = build_design(g, f)
    D = np.full(len(g), np.nan)
    src = d.frame["source_row"].to_numpy(int)
    D[src - 1] = d.frame["D"].to_numpy(float)
    t = pd.to_datetime(g["state_time"], utc=True)
    contig = np.r_[False, (t.diff().iloc[1:] == vt.BAR).to_numpy()]
    absD = np.abs(D)
    dabs = np.where(contig, absD - np.r_[np.nan, absD[:-1]], np.nan)
    return pd.DataFrame({"absD": absD, "D": D, "d_absD": dabs})


def raw_features(g, nuis):
    raw, used, _ = fi.raw_feature_frame(g)
    return pd.concat([raw.reset_index(drop=True), d_features(g, nuis)], axis=1), used


def eligibility(mat, used_specs, train_rows, h):
    X = mat.X[h]
    cov = X.loc[train_rows].notna().mean()
    nun = X.loc[train_rows].nunique(dropna=True)
    avail = {c for c in X.columns if cov.get(c, 0.0) >= MIN_FEATURE_COVERAGE and nun.get(c, 0) >= 3}
    specs_ok = [s for s in used_specs if s.name in avail]
    return avail, fi.block_map(specs_ok), cov


def prepare(asset, train_sessions, frame=None):
    g = asset_frame(asset, frame)
    panel = build_panel_v02(g)
    nuis = vh.fit_nuisance(g, panel, train_sessions)
    raw, used = raw_features(g, nuis)
    mat = vh.materialize(panel, nuis, raw)
    return dict(g=g, panel=panel, nuis=nuis, raw=raw, used=used, mat=mat)


INPUTS = {a: prepare(a, SCREEN_TRAIN) for a in ASSETS}

for a, b in INPUTS.items():
    vt.alignment_selftest(b["g"])
print("Alignment self-test passed for", ", ".join(INPUTS))

cov_rows = []
for a, b in INPUTS.items():
    m, P = b["mat"], b["panel"]
    for h in HORIZONS:
        dev = np.isin(P.session_date, DEV_SESSIONS)
        h_ok = np.isfinite(m.H[h][vh.H_NAMES].to_numpy(float)).all(axis=1)
        cov_rows.append({
            "underlying": a, "horizon": h, "origins": int(dev.sum()),
            "target_share": float(np.isfinite(m.y[h][dev]).mean()),
            "HAR_share": float(h_ok[dev].mean()),
            "W_imp_share": float(np.isfinite(m.X[h]["log_W_imp_alloc"].to_numpy()[dev]).mean()),
            "D_share": float(np.isfinite(b["raw"]["D"].to_numpy()[dev]).mean()),
            "log10_eps": np.log10(b["nuis"].eps_target[h]),
            "iv_divisor": P.iv_divisor,
            "median_iv": float(np.nanmedian(P.iv)),
        })
print("Target and input coverage (development sessions):")
display(pd.DataFrame(cov_rows).round(4))

Alignment self-test passed for AVAX, BTC, ETH, HYPE, SOL, XRP
Target and input coverage (development sessions):


,underlying,horizon,origins,target_share,HAR_share,W_imp_share,D_share,log10_eps,iv_divisor,median_iv
0,AVAX,30m,4608,0.8848,0.8116,0.9125,0.7719,-5.4438,100.0,0.6607
1,AVAX,60m,4608,0.8060,0.8116,0.9084,0.7719,-4.9108,100.0,0.6607
2,BTC,30m,4608,0.8848,0.8116,0.9143,0.7730,-6.8011,100.0,0.2808
3,BTC,60m,4608,0.8060,0.8116,0.9123,0.7730,-6.2319,100.0,0.2808
4,ETH,30m,4608,0.8848,0.8116,0.9201,0.7730,-6.4400,100.0,0.3635
5,ETH,60m,4608,0.8060,0.8116,0.9186,0.7730,-5.9360,100.0,0.3635
6,HYPE,30m,4608,0.8848,0.8116,0.9067,0.7730,-5.9636,100.0,0.5748
7,HYPE,60m,4608,0.8060,0.8116,0.9056,0.7730,-5.4766,100.0,0.5748
8,SOL,30m,4608,0.8848,0.8116,0.8635,0.7728,-5.9160,100.0,0.5198
9,SOL,60m,4608,0.8060,0.8116,0.8631,0.7728,-5.3490,100.0,0.5198


## 3. Target diagnostics

Overlap makes consecutive 30m / 60m targets strongly autocorrelated; inference is session-blocked throughout. The terminal target is compared with its log-return analogue (004 §3) and with the clock-allocated implied variance of the settlement average.

In [4]:
diag = []
for a, b in INPUTS.items():
    m, P = b["mat"], b["panel"]
    for h in HORIZONS:
        y = pd.Series(m.y[h])
        lag = vt.HORIZONS[h]
        diag.append({
            "underlying": a, "horizon": h, "n": int(y.notna().sum()),
            "mean": y.mean(), "sd": y.std(), "skew": y.skew(),
            "ac_lag1": y.autocorr(1), f"ac_lag_m": y.autocorr(lag),
            "corr_y_l30m": pd.Series(m.y[h]).corr(m.H[h]["l30m"]),
        })
print("Log-RV targets:")
display(pd.DataFrame(diag).round(3))

term = []
for a, b in INPUTS.items():
    s = b["mat"].settle
    ok = np.isfinite(s.rv_settle) & (s.rv_settle > 0)
    w = ok & np.isfinite(s.W_imp_settle) & (s.W_imp_settle > 0)
    term.append({
        "underlying": a,
        "finite_share": float(ok.mean()),
        "median_level_over_log": float(np.nanmedian(s.rv_settle[ok] / s.rv_settle_log[ok])),
        "p99_abs_log_diff": float(np.nanquantile(np.abs(np.log(s.rv_settle[ok] / s.rv_settle_log[ok])), 0.99)),
        "corr_log_rv_settle_log_W": float(np.corrcoef(np.log(s.rv_settle[w]), np.log(s.W_imp_settle[w]))[0, 1]) if w.sum() > 10 else np.nan,
        "median_log_rv_over_W": float(np.nanmedian(np.log(s.rv_settle[w] / s.W_imp_settle[w]))) if w.any() else np.nan,
    })
print("Provisional terminal target (descriptive only):")
display(pd.DataFrame(term).round(4))

Log-RV targets:


,underlying,horizon,n,mean,sd,skew,ac_lag1,ac_lag_m,corr_y_l30m
0,AVAX,30m,11458,-10.574,0.914,0.528,0.923,0.509,0.509
1,AVAX,60m,10034,-9.667,0.769,0.636,0.972,0.574,0.527
2,BTC,30m,11457,-12.620,1.167,0.177,0.936,0.591,0.590
3,BTC,60m,10033,-11.762,1.023,0.284,0.977,0.652,0.605
4,ETH,30m,11458,-12.031,1.130,0.352,0.934,0.568,0.565
5,ETH,60m,10034,-11.178,1.007,0.415,0.975,0.606,0.570
6,HYPE,30m,11457,-11.036,1.010,0.403,0.925,0.526,0.523
7,HYPE,60m,10033,-10.194,0.879,0.553,0.973,0.609,0.551
8,SOL,30m,11457,-11.391,0.966,0.533,0.925,0.516,0.510
9,SOL,60m,10033,-10.491,0.812,0.688,0.972,0.570,0.523


Provisional terminal target (descriptive only):


,underlying,finite_share,median_level_over_log,p99_abs_log_diff,corr_log_rv_settle_log_W,median_log_rv_over_W
0,AVAX,0.2683,0.9995,0.0909,0.8201,0.0140
1,BTC,0.2684,1.0005,0.0486,0.7999,0.1321
2,ETH,0.2684,1.0007,0.0509,0.7770,0.0366
3,HYPE,0.2684,0.9998,0.0512,0.8819,-0.1363
4,SOL,0.2684,1.0000,0.0650,0.7884,-0.1234
5,XRP,0.2684,0.9998,0.0588,0.7651,-0.0579


## 4. Development screening — corrected nested tests

The original v0.1 run is retained in git history but its M2a–M4 comparisons are not interpreted.

Design v0.2 fixes both identified flaws:

1. **Paired one-SE penalty selection.** For each candidate lambda, uncertainty is the standard error across validation sessions of the paired loss difference versus the best lambda.
2. **Pure nested increments.** For each pair A vs B, both are trained on A's complete rows. Penalties already selected for B's blocks remain frozen. Only the new block is tuned.

The screening window remains development-only: fit on 2026-09-22 through 09-28 and evaluate on 09-29 through 10-07.

In [5]:
NESTED_STAGES = [("M2a", "M1"), ("M2b", "M2a"), ("M3", "M2b"), ("M4", "M3")]


def _har_matrix(mat, h, ok):
    Hdf = mat.H[h].loc[ok, vh.H_NAMES]
    keep = [c for c in vh.H_NAMES if c == "const" or float(Hdf[c].std()) > 1e-12]
    Hm = Hdf[keep].to_numpy(float)
    while len(keep) > 1 and np.linalg.matrix_rank(Hm) < Hm.shape[1]:
        keep = keep[:-1]
        Hm = Hdf[keep].to_numpy(float)
    return keep, Hm


def fit_block_ridge(mat, h, model, blocks, rows):
    """HAR/clock unpenalized; each added block has its own frozen Ridge penalty."""
    x_names, penalties = [], []
    for _, cols, lam in blocks:
        for c in cols:
            if c not in x_names:
                x_names.append(c)
                penalties.append(float(lam))

    if not x_names:
        return vh.fit_vol(mat, h, model, [], rows, lam=0.0)

    ok = vh.complete(mat, h, x_names, rows)
    keep, Hm = _har_matrix(mat, h, ok)
    y = mat.y[h][ok]
    n = len(y)
    if n <= Hm.shape[1] + len(x_names) + 5:
        raise ValueError(f"{mat.asset} {h} {model}: too few common rows ({n}).")

    Xraw = mat.X[h].loc[ok, x_names].to_numpy(float)
    mu, sd = Xraw.mean(axis=0), Xraw.std(axis=0)
    use = sd > 1e-12
    x_names = [c for c, u in zip(x_names, use) if u]
    pen = np.asarray(penalties, float)[use]
    mu, sd = mu[use], sd[use]
    Xs = np.clip((Xraw[:, use] - mu) / sd, -vh.CLIP_Z, vh.CLIP_Z)

    Q, _ = np.linalg.qr(Hm)
    yr = y - Q @ (Q.T @ y)
    Xr = Xs - Q @ (Q.T @ Xs)

    A = Xr.T @ Xr / n + np.diag(pen)
    b = Xr.T @ yr / n
    try:
        gamma = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        gamma, *_ = np.linalg.lstsq(A, b, rcond=None)

    phi, *_ = np.linalg.lstsq(Hm, y - Xs @ gamma, rcond=None)
    resid = y - Hm @ phi - Xs @ gamma

    fv = vh.FittedVol(
        model=model,
        horizon=h,
        kind="ridge",
        h_names=keep,
        x_names=list(x_names),
        x_mean=mu.tolist(),
        x_scale=sd.tolist(),
        phi=np.asarray(phi).tolist(),
        gamma=np.asarray(gamma).tolist(),
        lam=float(max(pen) if len(pen) else 0.0),
        l1_ratio=0.0,
    )
    return vh._finish(fv, resid)


def paired_one_se_choice(records):
    """Largest lambda within one paired-session SE of the best lambda."""
    d = pd.DataFrame(records)
    if d.empty:
        return np.nan, pd.DataFrame()

    piv = d.pivot_table(index=["fold", "session"], columns="lam", values="qlike", aggfunc="mean")
    means = piv.mean(axis=0)
    best_lam = float(means.idxmin())
    rows = []

    best_series = piv[best_lam]
    for lam in sorted(piv.columns):
        cand_series = piv[lam]
        common = best_series.notna() & cand_series.notna()
        diff = cand_series.loc[common] - best_series.loc[common]
        mean_diff = float(diff.mean()) if len(diff) else np.nan
        se_diff = float(diff.std(ddof=1) / np.sqrt(len(diff))) if len(diff) > 1 else 0.0
        rows.append({
            "lam": float(lam),
            "mean_qlike": float(means.loc[lam]),
            "mean_diff_vs_best": mean_diff,
            "se_diff": se_diff,
            "n_sessions": int(len(diff)),
            "within_1se": bool(np.isfinite(mean_diff) and mean_diff <= se_diff + 1e-15),
        })

    tab = pd.DataFrame(rows)
    within = tab[tab["within_1se"]]
    chosen = float(within["lam"].max()) if len(within) else best_lam
    return chosen, tab


def select_new_block_lambda(fold_materials, h, base_blocks, new_cols, lam_grid=vh.LAMBDA_GRID):
    if not new_cols:
        return 0.0, pd.DataFrame()

    recs = []
    for fold, (tr_s, va_s, mat) in enumerate(fold_materials):
        tr = np.isin(mat.sessions, sorted(tr_s))
        va = np.isin(mat.sessions, sorted(va_s))
        large_cols = list(dict.fromkeys([c for _, cols, _ in base_blocks for c in cols] + list(new_cols)))
        common_tr = tr & vh.complete(mat, h, large_cols)
        common_va = va & vh.complete(mat, h, large_cols)

        for lam in lam_grid:
            try:
                fv = fit_block_ridge(
                    mat, h, "candidate",
                    base_blocks + [("new", list(new_cols), float(lam))],
                    common_tr,
                )
            except ValueError:
                continue

            sc = vh.row_scores(fv, mat, h)
            sm = vh.session_mean(sc["qlike"].to_numpy(), mat.sessions, common_va)
            for s, v in sm.items():
                recs.append({"fold": fold, "session": s, "lam": float(lam), "qlike": float(v)})

    return paired_one_se_choice(recs)


def oof_m1_predictions(fold_materials, h):
    out = []
    for tr_s, va_s, mat in fold_materials:
        tr = np.isin(mat.sessions, sorted(tr_s))
        va = np.isin(mat.sessions, sorted(va_s))
        try:
            fv = vh.fit_vol(mat, h, "M1", [], tr)
        except ValueError:
            continue
        sc = vh.row_scores(fv, mat, h)
        r = va & vh.complete(mat, h, []) & np.isfinite(sc["yhat"].to_numpy())
        out.append(pd.DataFrame({
            "y": mat.y[h][r],
            "yhat": sc["yhat"].to_numpy()[r],
            "session": mat.sessions[r],
        }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def fit_window(asset, b, train_sessions, h):
    """Sequential block selection with pure nested-pair comparisons."""
    g, P, mat = b["g"], b["panel"], b["mat"]
    tr = np.isin(mat.sessions, sorted(train_sessions))
    avail, blocks_map, cov = eligibility(mat, b["used"], tr, h)
    specs = vh.model_specs(blocks_map, avail)
    folds = blocked_folds(sorted(train_sessions), n_folds=3, min_train_share=0.45)

    def build(trs):
        nuis = vh.fit_nuisance(g, P, trs)
        return vh.materialize(P, nuis, b["raw"])

    fold_materials = [(tr_s, va_s, build(tr_s)) for tr_s, va_s in folds]

    selected_lam, cv_tables, block_defs = {}, {}, []
    for large, small in NESTED_STAGES:
        small_cols = specs[small] or []
        large_cols = specs[large] or []
        new_cols = [c for c in large_cols if c not in small_cols]
        lam, tab = select_new_block_lambda(fold_materials, h, block_defs, new_cols)
        if not np.isfinite(lam):
            lam = max(vh.LAMBDA_GRID)
        selected_lam[large] = float(lam)
        cv_tables[large] = tab
        block_defs.append((large, new_cols, float(lam)))

    fits = {
        "M0": vh.fit_vol(mat, h, "M0", [], tr),
        "M1": vh.fit_vol(mat, h, "M1", [], tr),
    }
    oof = oof_m1_predictions(fold_materials, h)
    fits["M1c"] = vh.calibrate_m1(fits["M1"], oof)

    cumulative = []
    blocks_by_model = {"M1": []}
    for large, small in NESTED_STAGES:
        new_cols = [c for c in (specs[large] or []) if c not in (specs[small] or [])]
        cumulative = cumulative + [(large, new_cols, selected_lam[large])]
        blocks_by_model[large] = list(cumulative)
        fits[large] = fit_block_ridge(mat, h, large, cumulative, tr)

    all_cols = specs["M4"] or []
    fits["M4_ols"] = fit_block_ridge(mat, h, "M4_ols", [("all", all_cols, 0.0)], tr)

    pair_fits = {}
    common_m1 = tr & vh.complete(mat, h, [])
    pair_fits["M1__M0"] = {
        "larger": vh.fit_vol(mat, h, "M1_pair", [], common_m1),
        "smaller": vh.fit_vol(mat, h, "M0", [], common_m1),
    }

    for large, small in NESTED_STAGES:
        common = tr & vh.complete(mat, h, specs[large] or [])
        pair_fits[f"{large}__{small}"] = {
            "larger": fit_block_ridge(mat, h, f"{large}_pair", blocks_by_model[large], common),
            "smaller": fit_block_ridge(mat, h, f"{small}_pair", blocks_by_model.get(small, []), common),
        }

    return dict(
        specs=specs,
        avail=avail,
        blocks=blocks_map,
        cv=cv_tables,
        lam=selected_lam,
        fits=fits,
        pair_fits=pair_fits,
        coverage=cov,
    )


SCREEN = {}
t_all = time.perf_counter()
for a, b in INPUTS.items():
    t0 = time.perf_counter()
    SCREEN[a] = {h: fit_window(a, b, SCREEN_TRAIN, h) for h in HORIZONS}
    print(f"{a}: v0.2 screening fits in {time.perf_counter() - t0:.0f}s")
print(f"total {time.perf_counter() - t_all:.0f}s")

lam_rows = []
for a, by_h in SCREEN.items():
    for h, r in by_h.items():
        lam_rows.append({
            "underlying": a,
            "horizon": h,
            **{f"lam_new_{m}": r["lam"].get(m, np.nan) for m in ("M2a", "M2b", "M3", "M4")},
            **{f"n_{m}": len(r["specs"][m]) for m in ("M2b", "M3", "M4")},
        })
print("Selected NEW-BLOCK penalties (paired one-SE) and model sizes:")
display(pd.DataFrame(lam_rows))

AVAX: v0.2 screening fits in 5s
BTC: v0.2 screening fits in 5s
ETH: v0.2 screening fits in 5s
HYPE: v0.2 screening fits in 5s
SOL: v0.2 screening fits in 5s
XRP: v0.2 screening fits in 5s
total 29s
Selected NEW-BLOCK penalties (paired one-SE) and model sizes:


,underlying,horizon,lam_new_M2a,lam_new_M2b,lam_new_M3,lam_new_M4,n_M2b,n_M3,n_M4
0,AVAX,30m,0.1,10000.0,10000.0,10000.0,3,36,65
1,AVAX,60m,0.0,10000.0,10000.0,10000.0,3,36,65
2,BTC,30m,10000.0,10000.0,10000.0,10000.0,3,36,63
3,BTC,60m,10000.0,10000.0,10000.0,10000.0,3,36,63
4,ETH,30m,10000.0,10000.0,10000.0,10000.0,3,36,65
5,ETH,60m,10000.0,10000.0,10000.0,10000.0,3,36,65
6,HYPE,30m,10000.0,10000.0,10000.0,10000.0,3,34,63
7,HYPE,60m,10000.0,10000.0,10000.0,10000.0,3,34,63
8,SOL,30m,10000.0,1.0,10000.0,10000.0,3,36,65
9,SOL,60m,10000.0,10000.0,10000.0,10000.0,3,36,65


In [6]:
SCORES, metrics = {}, []
for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))
    SCORES[a] = {}

    for h, r in by_h.items():
        SCORES[a][h] = {m: vh.row_scores(fv, mat, h) for m, fv in r["fits"].items()}
        y = mat.y[h]
        dy = y - mat.delta_base[h]
        ref = {k: SCORES[a][h][k]["yhat"].to_numpy() for k in ("M1", "M2b") if k in SCORES[a][h]}

        for m, sc in SCORES[a][h].items():
            yhat = sc["yhat"].to_numpy()
            rows = te & np.isfinite(sc["qlike"].to_numpy())
            rec = {
                "underlying": a,
                "horizon": h,
                "model": m,
                "n": int(rows.sum()),
                "qlike": float(sc["qlike"][rows].mean()),
                "log_rmse": float(np.sqrt(sc["sqerr"][rows].mean())),
                "ls_gauss": float(sc["ls_gauss"][rows].mean()),
                "ls_t": float(sc["ls_t"][rows].mean()),
                "r2": vh.r2(y[rows], yhat[rows]),
                "r2_change_form": vh.r2(dy[rows], (yhat - mat.delta_base[h])[rows]),
            }
            for k, yr in ref.items():
                both = rows & np.isfinite(yr)
                rec[f"dR2_vs_{k}"] = vh.r2(y[both], yhat[both]) - vh.r2(y[both], yr[both])
            rec.update(vh.mz(y[rows], yhat[rows], mat.sessions[rows], n_boot=500))
            metrics.append(rec)

metrics = pd.DataFrame(metrics)
print("v0.2 screening metrics on 2026-09-29 -> 10-07 (descriptive; hierarchy tested below with pair-specific fits):")
display(metrics.round(4))

metrics.to_csv(OUT / "06_screening_metrics.csv", index=False)


v0.2 screening metrics on 2026-09-29 -> 10-07 (descriptive; hierarchy tested below with pair-specific fits):


,underlying,horizon,model,n,qlike,log_rmse,ls_gauss,ls_t,r2,r2_change_form,dR2_vs_M1,dR2_vs_M2b,mz_a,mz_b,mz_b_lo,mz_b_hi
0,AVAX,30m,M0,2094,0.4556,0.9034,-1.3384,-1.3336,0.1271,-0.0001,-0.0796,-0.0409,-4.5724,0.5646,0.3876,0.6575
1,AVAX,30m,M1,1860,0.4225,0.8592,-1.2731,-1.2824,0.2344,0.0941,0.0000,0.0194,7.5644,1.7741,1.4122,2.0139
2,AVAX,30m,M1c,1860,0.5175,0.9837,-1.4267,-1.4370,-0.0035,-0.1873,-0.2379,-0.2078,36.3068,4.6369,3.6911,5.2636
3,AVAX,30m,M2a,1726,0.4253,0.8768,-1.3027,-1.3121,0.2132,0.0625,-0.0179,0.0029,6.4183,1.6675,1.2802,1.8581
4,AVAX,30m,M2b,1630,0.4339,0.8840,-1.3124,-1.3221,0.2125,0.0488,-0.0194,0.0000,6.5952,1.6843,1.2758,1.8766
5,AVAX,30m,M3,933,0.4128,0.8664,-1.2954,-1.3065,0.1776,0.1373,-0.0861,-0.0614,8.6486,1.8899,1.4803,2.1645
6,AVAX,30m,M4,631,0.4302,0.8510,-1.2882,-1.2852,0.2282,0.1882,-0.0668,-0.0551,8.2103,1.8239,1.3593,2.2283
7,AVAX,30m,M4_ols,631,0.6566,1.0840,-1.9840,-1.8836,-0.2522,-0.3172,-0.5473,-0.5355,-6.2768,0.3927,0.1499,0.5898
8,AVAX,60m,M0,1901,0.3042,0.7351,-1.1303,-1.1257,0.2020,-0.0717,-0.0493,0.0412,-3.4940,0.6459,0.3992,0.7963
9,AVAX,60m,M1,1690,0.2918,0.7200,-1.1187,-1.1269,0.2658,0.0054,0.0000,0.0606,5.9827,1.6745,1.2928,1.9507


### Pure paired comparisons

Positive means the larger model is better.

Every hierarchy step now uses pair-specific restricted and full fits with:

- identical training rows;
- identical nuisance fit;
- identical penalties on all previously admitted blocks;
- only the newly added block differing.

The non-overlapping view keeps session-aligned forecast origins every h.

In [7]:
PAIR_KEYS = [("M1", "M0")] + NESTED_STAGES
cmp_rows = []

for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))

    for h, r in by_h.items():
        nov = vh.non_overlapping(mat.tte_min, h)

        for large, small in PAIR_KEYS:
            pf = r["pair_fits"][f"{large}__{small}"]
            sa = vh.row_scores(pf["larger"], mat, h)
            sb = vh.row_scores(pf["smaller"], mat, h)

            for view, rows in (("all", te), ("non_overlapping", te & nov)):
                c = vh.compare(sa, sb, mat.sessions, rows, n_boot=N_BOOT)
                cmp_rows.append({
                    "underlying": a,
                    "horizon": h,
                    "model": large,
                    "vs": small,
                    "view": view,
                    **c,
                })

        sa = SCORES[a][h]["M1c"]
        sb = SCORES[a][h]["M1"]
        for view, rows in (("all", te), ("non_overlapping", te & nov)):
            c = vh.compare(sa, sb, mat.sessions, rows, n_boot=N_BOOT)
            cmp_rows.append({
                "underlying": a,
                "horizon": h,
                "model": "M1c",
                "vs": "M1",
                "view": view,
                **c,
            })

comparisons = pd.DataFrame(cmp_rows)
show = [
    "underlying", "horizon", "model", "vs", "view", "n_rows",
    "qlike_n_sessions", "qlike_mean_session_diff", "qlike_boot_lo", "qlike_boot_hi",
    "qlike_sessions_won", "sqerr_mean_session_diff", "ls_t_mean_session_diff",
]
display(comparisons[show].round(4))

comparisons.to_csv(OUT / "06_screening_comparisons.csv", index=False)
print("Saved notebooks/HSMM/06/06_screening_metrics.csv and 06_screening_comparisons.csv")


,underlying,horizon,model,vs,view,n_rows,qlike_n_sessions,qlike_mean_session_diff,qlike_boot_lo,qlike_boot_hi,qlike_sessions_won,sqerr_mean_session_diff,ls_t_mean_session_diff
0,AVAX,30m,M1,M0,all,1860,9,7.0804,-15.7294,29.3501,0.5556,15.8687,10.7857
1,AVAX,30m,M1,M0,non_overlapping,319,9,0.7223,-3.4667,5.5550,0.4444,3.9269,2.1237
2,AVAX,30m,M2a,M1,all,1726,9,-2.9359,-6.6169,0.8624,0.4444,-9.7368,-9.2230
3,AVAX,30m,M2a,M1,non_overlapping,296,9,-0.5413,-1.1444,0.0660,0.4444,-1.6540,-1.6057
4,AVAX,30m,M2b,M2a,all,1630,9,0.0000,0.0000,0.0000,0.7778,0.0000,0.0000
5,AVAX,30m,M2b,M2a,non_overlapping,283,9,0.0000,-0.0000,0.0000,0.6667,0.0000,0.0000
6,AVAX,30m,M3,M2b,all,933,9,0.0006,-0.0001,0.0016,0.6667,0.0008,0.0003
7,AVAX,30m,M3,M2b,non_overlapping,163,9,0.0000,-0.0001,0.0001,0.4444,0.0000,-0.0001
8,AVAX,30m,M4,M3,all,631,9,-0.0007,-0.0018,0.0000,0.3333,-0.0014,-0.0016
9,AVAX,30m,M4,M3,non_overlapping,110,9,-0.0002,-0.0004,-0.0000,0.2222,-0.0003,-0.0003


Saved notebooks/HSMM/06/06_screening_metrics.csv and 06_screening_comparisons.csv


### Population retained by each model (004 §11)

Option-based models score only quote-backed rows outside the settlement window. This table shows what population remains.

In [8]:
def tte_bucket(tte):
    return pd.cut(tte / 60.0, [-1, 4, 8, 16, 24.5], labels=["0-4h", "4-8h", "8-16h", "16-24h"])

sel = []
for a, by_h in SCREEN.items():
    b = INPUTS[a]
    mat, raw = b["mat"], b["raw"]
    dev = np.isin(mat.sessions, DEV_SESSIONS)
    spread = raw["log_option_spread_med"].to_numpy() if "log_option_spread_med" in raw else np.full(len(raw), np.nan)
    for h, r in by_h.items():
        elig = dev & np.isfinite(mat.y[h]) & np.isfinite(mat.H[h][vh.H_NAMES].to_numpy(float)).all(axis=1)
        for m in ("M1", "M2a", "M2b", "M3", "M4"):
            keep = elig & vh.complete(mat, h, r["specs"][m])
            rec = {"underlying": a, "horizon": h, "model": m, "eligible": int(elig.sum()), "retained": int(keep.sum()),
                   "retained_share": float(keep.sum() / max(elig.sum(), 1)),
                   "weekend_share_retained": float(INPUTS[a]["panel"].weekend[keep].mean()) if keep.any() else np.nan,
                   "min_session_share": float(pd.Series(keep[elig]).groupby(mat.sessions[elig]).mean().min()),
                   "median_y_retained": float(np.nanmedian(mat.y[h][keep])) if keep.any() else np.nan,
                   "median_y_excluded": float(np.nanmedian(mat.y[h][elig & ~keep])) if (elig & ~keep).any() else np.nan,
                   "median_log_opt_spread_retained": float(np.nanmedian(spread[keep])) if keep.any() else np.nan,
                   "median_log_opt_spread_excluded": float(np.nanmedian(spread[elig & ~keep])) if (elig & ~keep).any() else np.nan}
            buckets = pd.Series(keep[elig]).groupby(tte_bucket(mat.tte_min[elig]).astype(str)).mean()
            rec.update({f"share_{k}": v for k, v in buckets.items()})
            sel.append(rec)
selection = pd.DataFrame(sel)
display(selection.round(4))

,underlying,horizon,model,eligible,retained,retained_share,weekend_share_retained,min_session_share,median_y_retained,median_y_excluded,median_log_opt_spread_retained,median_log_opt_spread_excluded,share_0-4h,share_16-24h,share_4-8h,share_8-16h
0,AVAX,30m,M1,3431,3431,1.0000,0.2778,1.0000,-10.2641,NaN,8.7409,NaN,1.0000,1.0000,1.0000,1.0000
1,AVAX,30m,M2a,3431,3181,0.9271,0.2663,0.8538,-10.2448,-10.4766,8.7139,9.3329,0.6763,0.9645,0.9738,0.9940
2,AVAX,30m,M2b,3431,3017,0.8793,0.2652,0.7602,-10.2488,-10.3961,8.7169,9.0706,0.6201,0.8744,0.9360,0.9803
3,AVAX,30m,M3,3431,1858,0.5415,0.2535,0.0000,-10.0983,-10.5411,8.7294,8.7516,0.3680,0.5856,0.6148,0.5483
4,AVAX,30m,M4,3431,1249,0.3640,0.2258,0.0000,-10.0631,-10.4235,8.6042,8.8662,0.0477,0.4843,0.2616,0.4816
5,AVAX,60m,M1,3148,3148,1.0000,0.2843,1.0000,-9.3985,NaN,8.7474,NaN,1.0000,1.0000,1.0000,1.0000
6,AVAX,60m,M2a,3148,2975,0.9450,0.2736,0.8736,-9.3934,-9.5389,8.7247,9.3329,0.7495,0.9696,0.9734,0.9936
7,AVAX,60m,M2b,3148,2832,0.8996,0.2722,0.7919,-9.3969,-9.4236,8.7281,9.0171,0.6957,0.8816,0.9379,0.9800
8,AVAX,60m,M3,3148,1744,0.5540,0.2615,0.0000,-9.2415,-9.6339,8.7406,8.7524,0.4079,0.5941,0.6154,0.5481
9,AVAX,60m,M4,3148,1175,0.3733,0.2340,0.0000,-9.1603,-9.5288,8.6249,8.8727,0.0538,0.4927,0.2663,0.4828


### Coefficients

HAR + clock coefficients of M1 (persistence structure), and the largest standardized added-block coefficients of M3 / M4.

In [9]:
har = []
top = []
for a, by_h in SCREEN.items():
    for h, r in by_h.items():
        m1 = r["fits"]["M1"]
        har.append({"underlying": a, "horizon": h, **dict(zip(m1.h_names, np.round(m1.phi, 4))), "smear": m1.smear})
        for m in ("M3", "M4"):
            fv = r["fits"][m]
            if fv.x_names:
                s = pd.Series(fv.gamma, index=fv.x_names)
                for name, v in s.reindex(s.abs().sort_values(ascending=False).index).head(10).items():
                    top.append({"underlying": a, "horizon": h, "model": m, "feature": name, "coef": v})
print("M1 HAR + clock coefficients:")
display(pd.DataFrame(har))
print("Largest added-block coefficients (standardized, after shrinkage):")
display(pd.DataFrame(top).round(4))

M1 HAR + clock coefficients:


,underlying,horizon,const,l5m,l30m,l1h,l4h,clock_fwd,clock_back,weekend,smear
0,AVAX,30m,-6.5964,0.0207,0.0386,0.0834,0.2629,0.9535,-0.3847,-0.0436,1.430286
1,AVAX,60m,-6.0068,0.0185,0.0015,0.0972,0.2626,0.9464,-0.3500,-0.0419,1.230700
2,BTC,30m,-7.5258,0.0055,0.1729,-0.0104,0.2698,1.0661,-0.3668,0.3741,1.403764
3,BTC,60m,-7.6474,0.0099,0.0301,0.0855,0.2329,1.2473,-0.4734,0.4311,1.225831
4,ETH,30m,-7.7815,0.0155,0.2011,0.0461,0.0881,0.6516,-0.0150,0.0939,1.410047
5,ETH,60m,-8.2486,0.0192,0.1125,0.0282,0.0743,0.8589,-0.1337,0.1041,1.258113
6,HYPE,30m,-7.2065,0.0086,0.1899,-0.0910,0.2746,1.0230,-0.6013,-0.1862,1.332488
7,HYPE,60m,-7.4431,0.0048,0.0823,-0.0846,0.2852,0.9460,-0.5074,-0.2266,1.182980
8,SOL,30m,-6.8234,0.0177,0.0363,0.0115,0.3967,0.7698,-0.1266,-0.0264,1.313761
9,SOL,60m,-6.3939,0.0132,-0.0250,0.0714,0.3628,0.8077,-0.1642,-0.0169,1.160752


Largest added-block coefficients (standardized, after shrinkage):


,underlying,horizon,model,feature,coef
0,AVAX,30m,M3,log_W_imp_alloc,-0.1346
1,AVAX,30m,M3,D,0.0000
2,AVAX,30m,M3,log_book_updates,0.0000
3,AVAX,30m,M3,log_avg_trade,-0.0000
4,AVAX,30m,M3,log_gk30,0.0000
5,AVAX,30m,M3,log_range_bps,0.0000
6,AVAX,30m,M3,log_trade_count,0.0000
7,AVAX,30m,M3,obi_20_min,-0.0000
8,AVAX,30m,M3,log_park30,0.0000
9,AVAX,30m,M3,microprice_mid_bps,-0.0000


### Physical forecast vs clock-allocated implied variance (descriptive)

$\log(\widehat W^{real}/\widetilde W^{imp})$ against $\log(RV/\widetilde W^{imp})$ on the evaluation rows. The median realized-to-implied log ratio is the in-sample variance-risk-premium proxy. This is not a P&L statement: a gap can be a rational premium (004 §10).

In [10]:
econ = []
for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))
    for h, r in by_h.items():
        lW = mat.X[h]["log_W_imp_alloc"].to_numpy()
        real = np.log(mat.level[h]) - lW
        for m in ("M1", "M2b", "M3"):
            sc = SCORES[a][h][m]
            gap_hat = sc["yhat"].to_numpy() + np.log(r["fits"][m].smear) - lW
            ok = te & np.isfinite(gap_hat) & np.isfinite(real)
            econ.append({"underlying": a, "horizon": h, "model": m, "n": int(ok.sum()),
                         "corr_gap_hat_gap_real": float(np.corrcoef(gap_hat[ok], real[ok])[0, 1]) if ok.sum() > 10 else np.nan,
                         "median_log_rv_over_W": float(np.median(real[ok])) if ok.any() else np.nan})
display(pd.DataFrame(econ).round(4))

,underlying,horizon,model,n,corr_gap_hat_gap_real,median_log_rv_over_W
0,AVAX,30m,M1,1726,0.3668,0.0206
1,AVAX,30m,M2b,1630,0.3834,0.0144
2,AVAX,30m,M3,933,0.2751,0.1769
3,AVAX,60m,M1,1599,0.4158,0.2256
4,AVAX,60m,M2b,1515,0.4277,0.2332
5,AVAX,60m,M3,873,0.3235,0.3618
6,BTC,30m,M1,1736,0.3824,-0.1031
7,BTC,30m,M2b,1677,0.3852,-0.1077
8,BTC,30m,M3,1548,0.3667,-0.1112
9,BTC,60m,M1,1612,0.3918,0.0497


## 5. Freeze the v0.2 specification

All nuisance quantities, eligibility, sequential new-block penalties, general model fits and pair-specific restricted/full fits are refitted on all development sessions (2026-09-22 through 2026-10-07).

The old 06_frozen_spec.json is design v0.1 and must not be used for confirmation. This notebook writes 06_frozen_spec_v02.json.

Commit the v0.2 file before scoring any post-2026-10-07 confirmation session.

In [11]:
def git_commit():
    try:
        return subprocess.check_output(
            ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
            text=True,
        ).strip()
    except Exception:
        return "unknown"


if MODE == "development" and FREEZE:
    if SPEC_PATH.exists() and not REFREEZE:
        print(f"{SPEC_PATH.name} exists: not overwritten (set NB12_REFREEZE=1 to replace it before confirmation).")
    else:
        spec = {
            "meta": {
                "design_version": DESIGN_VERSION,
                "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "git_commit": git_commit(),
                "vol_targets": vt.VOL_TARGETS_VERSION,
                "vol_har": vh.VOL_HAR_VERSION,
                "clock_baseline": CLOCK_BASELINE_VERSION,
                "full_information": fi.FULL_INFORMATION_VERSION,
                "development_sessions": DEV_SESSIONS,
                "confirmation_starts_after": DEV_END,
                "confirmation_min_sessions": CONFIRM_MIN_SESSIONS,
                "min_feature_coverage": MIN_FEATURE_COVERAGE,
                "backward_min_coverage": BACK_MIN_COVERAGE,
                "lambda_grid": list(vh.LAMBDA_GRID),
                "horizons": vt.HORIZONS,
                "primary_score": "QLIKE (Duan smearing)",
                "hierarchy": [list(p) for p in NESTED_STAGES],
                "nested_test": "common rows; paired-difference one-SE; frozen prior-block penalties",
            },
            "assets": {},
        }

        t0 = time.perf_counter()
        FROZEN_INPUTS = {}

        for a in ASSETS:
            b = prepare(a, set(DEV_SESSIONS))
            FROZEN_INPUTS[a] = b
            entry = {"nuisance": b["nuis"].to_dict(), "horizons": {}}

            for h in HORIZONS:
                r = fit_window(a, b, set(DEV_SESSIONS), h)
                entry["horizons"][h] = {
                    "specs": r["specs"],
                    "block_lambda": r["lam"],
                    "models": {m: fv.to_dict() for m, fv in r["fits"].items()},
                    "pair_models": {
                        key: {side: fv.to_dict() for side, fv in pair.items()}
                        for key, pair in r["pair_fits"].items()
                    },
                }

            spec["assets"][a] = entry
            print(f"{a}: v0.2 frozen ({time.perf_counter() - t0:.0f}s)")

        SPEC_PATH.write_text(json.dumps(spec, indent=1, default=float))
        print(
            f"Wrote {SPEC_PATH} ({SPEC_PATH.stat().st_size / 1e6:.2f} MB). "
            "Commit it before any confirmation scoring."
        )

## 6. Confirmation — binding after 30 completed sessions

Run with NB12_MODE=confirmation.

Only 06_frozen_spec_v02.json is accepted. The notebook checks the design version before scoring. The v0.1 frozen file is intentionally ignored.

Each hierarchy step uses the stored pair-specific restricted/full models, preserving identical training rows and the frozen penalties for all previously admitted blocks.

With fewer than 30 completed sessions the readout is a non-binding checkpoint.

In [12]:
if MODE == "confirmation":
    if not SPEC_PATH.exists():
        raise RuntimeError(
            f"{SPEC_PATH.name} not found. Re-run development mode under design v{DESIGN_VERSION} "
            "and commit the new frozen spec before confirmation."
        )

    spec = json.loads(SPEC_PATH.read_text())
    if spec.get("meta", {}).get("design_version") != DESIGN_VERSION:
        raise RuntimeError(
            f"Frozen spec design {spec.get('meta', {}).get('design_version')} != required {DESIGN_VERSION}."
        )

    print(
        f"Frozen v{DESIGN_VERSION} spec from commit {spec['meta']['git_commit'][:10]} "
        f"created {spec['meta']['created_utc']}"
    )

    conf_sessions = sorted(
        s for s in df["session_date"].unique()
        if s > spec["meta"]["confirmation_starts_after"]
        and pd.Timestamp(f"{s} 08:00", tz="UTC") <= df["state_time"].max()
    )
    binding = len(conf_sessions) >= spec["meta"]["confirmation_min_sessions"]
    print(
        f"Completed confirmation sessions: {len(conf_sessions)} -> "
        f"{'BINDING readout' if binding else 'NON-BINDING checkpoint'}"
    )

    conf_rows, conf_metrics = [], []

    for a in ASSETS:
        entry = spec["assets"][a]
        g = asset_frame(a)
        panel = build_panel_v02(g)
        nuis = vh.Nuisance(**entry["nuisance"])
        raw, _ = raw_features(g, nuis)
        mat = vh.materialize(panel, nuis, raw)
        cr = np.isin(mat.sessions, conf_sessions)

        for h, hz in entry["horizons"].items():
            general = {
                m: vh.row_scores(vh.FittedVol.from_dict(d), mat, h)
                for m, d in hz["models"].items()
            }

            for m, s in general.items():
                rows = cr & np.isfinite(s["qlike"].to_numpy())
                conf_metrics.append({
                    "underlying": a,
                    "horizon": h,
                    "model": m,
                    "n": int(rows.sum()),
                    "qlike": float(s["qlike"][rows].mean()) if rows.any() else np.nan,
                    "r2": vh.r2(mat.y[h][rows], s["yhat"].to_numpy()[rows]) if rows.any() else np.nan,
                })

            nov = vh.non_overlapping(mat.tte_min, h)

            for large, small in [("M1", "M0")] + NESTED_STAGES:
                key = f"{large}__{small}"
                pair = hz["pair_models"][key]
                sa = vh.row_scores(vh.FittedVol.from_dict(pair["larger"]), mat, h)
                sb = vh.row_scores(vh.FittedVol.from_dict(pair["smaller"]), mat, h)

                for view, rows in (("all", cr), ("non_overlapping", cr & nov)):
                    c = vh.compare(sa, sb, mat.sessions, rows, n_boot=N_BOOT)
                    conf_rows.append({
                        "underlying": a,
                        "horizon": h,
                        "model": large,
                        "vs": small,
                        "view": view,
                        **c,
                    })

            if "M1c" in general and "M1" in general:
                for view, rows in (("all", cr), ("non_overlapping", cr & nov)):
                    c = vh.compare(general["M1c"], general["M1"], mat.sessions, rows, n_boot=N_BOOT)
                    conf_rows.append({
                        "underlying": a,
                        "horizon": h,
                        "model": "M1c",
                        "vs": "M1",
                        "view": view,
                        **c,
                    })

    confirmation = pd.DataFrame(conf_rows)
    display(pd.DataFrame(conf_metrics).round(4))
    display(confirmation[show].round(4))
else:
    print("Development mode: confirmation not run.")

Frozen v0.2 spec from commit e1a3669acf created 2026-10-07T18:23:15+00:00
Completed confirmation sessions: 30 -> BINDING readout


,underlying,horizon,model,n,qlike,r2
0,AVAX,30m,M0,6233,0.5273,-0.0595
1,AVAX,30m,M1,5112,0.3876,0.2754
2,AVAX,30m,M1c,5112,0.3917,0.2537
3,AVAX,30m,M2a,4702,0.3893,0.2775
4,AVAX,30m,M2b,4355,0.3987,0.2825
5,AVAX,30m,M3,2369,0.3940,0.2261
6,AVAX,30m,M4,1724,0.4056,0.2265
7,AVAX,30m,M4_ols,1724,0.4651,0.1664
8,AVAX,60m,M0,5353,0.4077,-0.0297
9,AVAX,60m,M1,4478,0.2915,0.2899


,underlying,horizon,model,vs,view,n_rows,qlike_n_sessions,qlike_mean_session_diff,qlike_boot_lo,qlike_boot_hi,qlike_sessions_won,sqerr_mean_session_diff,ls_t_mean_session_diff
0,AVAX,30m,M1,M0,all,5112,30,23.3869,16.3688,30.6210,0.9333,51.4272,33.7825
1,AVAX,30m,M1,M0,non_overlapping,863,30,4.4918,3.2056,5.7057,0.9000,9.7775,6.0280
2,AVAX,30m,M2a,M1,all,4702,30,0.0000,-0.0000,0.0000,0.6667,0.0000,0.0000
3,AVAX,30m,M2a,M1,non_overlapping,789,30,0.0000,-0.0000,0.0000,0.7000,0.0000,0.0000
4,AVAX,30m,M2b,M2a,all,4355,30,-0.1992,-0.7392,0.1350,0.5000,0.0262,0.0596
5,AVAX,30m,M2b,M2a,non_overlapping,733,30,-0.0764,-0.2218,0.0234,0.4333,-0.0432,-0.0259
6,AVAX,30m,M3,M2b,all,2369,30,0.0001,-0.0000,0.0003,0.7000,-0.0000,-0.0001
7,AVAX,30m,M3,M2b,non_overlapping,401,29,0.0000,-0.0000,0.0001,0.6897,0.0000,-0.0000
8,AVAX,30m,M4,M3,all,1724,29,0.0007,-0.0007,0.0017,0.8276,0.0022,0.0020
9,AVAX,30m,M4,M3,non_overlapping,297,29,0.0001,-0.0001,0.0003,0.8276,0.0004,0.0003


## 7. Interpretation gates — v0.2

1. Targets and inputs must remain causal; the alignment self-test must pass.
2. Do not interpret the old v0.1 M2a–M4 results. Their penalty selection and training-row comparisons were flawed.
3. The binding sensor hierarchy is M2a vs M1, M2b vs M2a, M3 vs M2b, and M4 vs M3 using the pair-specific common-row fits.
4. A new block's selected penalty is informative only under the paired-difference one-SE rule implemented here.
5. Module 13 is independent of sensor success. Switching-HAR must later beat both raw M1 and training-only recalibrated M1c on common OOS rows.
6. No economic claim is made from forecast-vs-implied gaps alone.
7. The terminal target remains provisional until the settlement-average path-simulation layer is built.
8. M4 coverage remains a separate external-validity issue: a later compact or masked option block may be required for the final 0–4h window.